# 03 — QMAP-style sequence identity + Leiden clustering

Part of a 5-notebook peptide-clustering + consensus-split exploration — see
`README_clustering.md` in this folder for the full picture and run order. This
notebook implements method (d): global pairwise sequence identity (Needleman-Wunsch-
style global alignment, BLOSUM45) -> similarity graph -> Leiden community detection —
the same underlying method QMAP itself uses for its own published train/test split,
and the same one this repo already uses for the *existing* `split` column
(`pipeline/curation/07_build_final_and_split.py`, `src/soamp/data/splitting.py`) and
in `scripts/EDA/generate_leiden_train_folds.ipynb`. Reused here via the same
`qmap.toolkit.clustering` calls rather than reimplemented.

**Why this matters for coverage:** unlike methods (a)/(b)/(c), which all work
directly from `smiles` and therefore score every peptide uniformly, this method
works from the amino-acid-letter `sequence` column — which has no valid
representation at all for peptides containing DBAASP's own unresolved-residue
placeholder (`X`/`x`). This notebook computes and reports that coverage gap
explicitly, with counts, rather than silently dropping those peptides from the
dataset or guessing a sequence for them.


In [1]:
import time
from pathlib import Path

import pandas as pd
from qmap.toolkit.clustering import build_graph, leiden_community_detection


## Setup: locate the repo and load/build the shared peptide-level frame (same pattern as notebooks 01/02)

In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root (no pyproject.toml found above cwd)")


REPO_ROOT = find_repo_root(Path.cwd())
DATA_DIR = REPO_ROOT / "data"
CLUSTERING_DIR = DATA_DIR / "clustering"
CLUSTERING_DIR.mkdir(exist_ok=True)
VOTERS_PATH = CLUSTERING_DIR / "peptide_voters.parquet"

print("REPO_ROOT:", REPO_ROOT)


REPO_ROOT: /Users/lukajin/PycharmProjects/soamp


In [3]:
def build_base_peptide_frame() -> pd.DataFrame:
    classification_df = pd.read_csv(DATA_DIR / "mic_classification_dataset.csv")
    base = (
        classification_df
        .drop_duplicates(subset="peptide_id")[["peptide_id", "sequence", "smiles", "has_noncanonical"]]
        .reset_index(drop=True)
    )
    regression_df = pd.read_csv(DATA_DIR / "final_mic_regression_dataset.csv")
    bond_types = regression_df.drop_duplicates(subset="peptide_id")[["peptide_id", "bond_type"]]
    base = base.merge(bond_types, on="peptide_id", how="left")
    base["is_linear"] = base["bond_type"].fillna("none") == "none"
    return base


def load_peptide_voters() -> pd.DataFrame:
    base = build_base_peptide_frame()
    if VOTERS_PATH.exists():
        existing = pd.read_parquet(VOTERS_PATH)
        voter_cols = [c for c in existing.columns if c not in base.columns]
        base = base.merge(existing[["peptide_id", *voter_cols]], on="peptide_id", how="left")
    return base


def save_peptide_voters(df: pd.DataFrame) -> None:
    df.to_parquet(VOTERS_PATH, index=False)
    print(f"saved {VOTERS_PATH} ({len(df)} rows, columns: {list(df.columns)})")


peptide_voters = load_peptide_voters()
print(peptide_voters.shape)
peptide_voters.head()


(12371, 8)


,peptide_id,sequence,smiles,has_noncanonical,bond_type,is_linear,cluster_fingerprint,cluster_descriptor
0,10,LFIFFF,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,False,none,True,223.0,3.0
1,11,RVKRVWPLVIRTVIAGYNLYRAIKKK,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,False,none,True,1.0,2.0
2,12,RKRIHIGPGRAFYTT,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1cnc[nH]1)NC(=O)...,False,none,True,8.0,2.0
3,13,RRXXRF,CC(=O)N[C@@H](CCCN=C(N)N)C(=O)N[C@@H](CCCN=C(N...,True,none,True,270.0,3.0
4,14,GIWDTIKSMGKVFAGKILQNL,CC[C@H](C)[C@H](NC(=O)CN)C(=O)N[C@@H](Cc1c[nH]...,False,none,True,3.0,2.0


In [4]:
def report_clustering_diagnostics(df: pd.DataFrame, cluster_col: str, runtime_seconds: float,
                                    imbalance_fraction_flag: float = 0.5) -> None:
    total = len(df)
    scored = df[cluster_col].notna()
    n_scored = int(scored.sum())
    sizes = df.loc[scored, cluster_col].value_counts()
    print(f"[{cluster_col}] runtime: {runtime_seconds:.2f}s")
    print(f"[{cluster_col}] coverage: {n_scored}/{total} peptides scored ({n_scored / total:.1%})")
    print(f"[{cluster_col}] cluster count: {len(sizes)}")
    print(f"[{cluster_col}] cluster size distribution: min={sizes.min()}, median={sizes.median():.0f}, "
          f"max={sizes.max()}, mean={sizes.mean():.1f}")
    largest_frac = sizes.max() / n_scored
    flag = " <-- SEVERE IMBALANCE" if largest_frac > imbalance_fraction_flag else ""
    print(f"[{cluster_col}] largest cluster is {largest_frac:.1%} of scored peptides{flag}")


## Coverage / linearization logic

Three cases, decided per-peptide from the raw `sequence` column:

1. **Canonical** (`has_noncanonical=False`) -- used as-is, no assumption needed.
2. **Non-canonical but only case-variant** -- DBAASP encodes a D-form residue as the
   lowercase letter of its canonical counterpart (e.g. `KAaaKWAaKaAK`). `.upper()`ing
   these gives a valid 20-letter-alphabet sequence BLOSUM45/global-alignment can
   score. **Assumption this introduces:** collapses the D/L stereochemistry
   distinction -- reasonable here since global sequence identity has no
   stereochemistry concept to begin with (this is a simplification, not a data
   error), but worth remembering when interpreting this method's clusters.
3. **Contains `X`/`x`** -- DBAASP's own placeholder for a residue it couldn't
   resolve to a specific letter at all (`has_unresolved_placeholder`, see
   `src/soamp/curation/parse_dbaasp.py`). **No valid sequence representation
   exists** for these positions. Confirmed via `p2smi` (this repo's own
   sequence<->SMILES tool) having no SMILES-to-sequence reverse capability either --
   there's no better linearization available. These peptides get `cluster_qmap =
   NaN`, a genuine null vote, and are excluded from the identity graph entirely
   rather than guessed.


In [5]:
def classify_qmap_input(sequence: str) -> str | None:
    if "X" in sequence or "x" in sequence:
        return None  # unresolved placeholder residue -- no valid representation
    return sequence.upper()  # no-op for canonical sequences, case-fold for D-form-only


peptide_voters["qmap_input_sequence"] = peptide_voters["sequence"].apply(classify_qmap_input)
peptide_voters["qmap_scoreable"] = peptide_voters["qmap_input_sequence"].notna()


### Coverage table

Total scoreable vs. unscoreable, broken out by linear vs. non-linear (`is_linear`,
derived from `bond_type` in the setup cell above) and by canonical vs. non-canonical
-- the actual coverage gap this method has, not an assumed one.


In [6]:
coverage_table = (
    peptide_voters
    .assign(
        coverage=lambda d: d["qmap_scoreable"].map({True: "scoreable", False: "unscoreable (X/x placeholder)"}),
        linearity=lambda d: d["is_linear"].map({True: "linear", False: "non-linear (has bond_type)"}),
    )
    .groupby(["coverage", "linearity"])
    .size()
    .unstack(fill_value=0)
)
coverage_table["total"] = coverage_table.sum(axis=1)
coverage_table.loc["total"] = coverage_table.sum()
print(coverage_table)

print()
print(f"canonical (has_noncanonical=False): {(~peptide_voters['has_noncanonical']).sum()} -- all scoreable")
_nc = peptide_voters[peptide_voters["has_noncanonical"]]
print(f"non-canonical: {len(_nc)} total -- "
      f"{_nc['qmap_scoreable'].sum()} scoreable via case-fold linearization, "
      f"{(~_nc['qmap_scoreable']).sum()} unscoreable (contain X/x)")


linearity                      linear  non-linear (has bond_type)  total
coverage                                                                
scoreable                        9217                        1671  10888
unscoreable (X/x placeholder)    1093                         390   1483
total                           10310                        2061  12371

canonical (has_noncanonical=False): 9889 -- all scoreable
non-canonical: 2482 total -- 999 scoreable via case-fold linearization, 1483 unscoreable (contain X/x)


## Build the sequence-identity graph and run Leiden

Same parameters as `scripts/EDA/generate_leiden_train_folds.ipynb` (and the
production splitter in `src/soamp/data/splitting.py` /
`pipeline/curation/07_build_final_and_split.py`) for methodological consistency
with the rest of this project -- `threshold=0.60` identity, BLOSUM45, gap
open/extension 5/1. All exposed as adjustable constants.


In [7]:
# Adjustable parameters (kept consistent with this repo's existing QMAP usage)
IDENTITY_THRESHOLD = 0.60
SUBSTITUTION_MATRIX = "blosum45"
GAP_OPEN = 5
GAP_EXTENSION = 1
LEIDEN_N_ITERATIONS = -1
LEIDEN_SEED = 42

scoreable_voters = peptide_voters[peptide_voters["qmap_scoreable"]].reset_index()  # keep original positional index
sequences = scoreable_voters["qmap_input_sequence"].tolist()

_t0 = time.perf_counter()
graph, edgelist = build_graph(
    sequences, threshold=IDENTITY_THRESHOLD, matrix=SUBSTITUTION_MATRIX,
    gap_open=GAP_OPEN, gap_extension=GAP_EXTENSION, use_cache=True,
    show_progress=True, num_threads=None,
)
print(f"build_graph: {graph.vcount()} nodes, {graph.ecount()} edges, "
      f"{time.perf_counter() - _t0:.1f}s")

clusters = leiden_community_detection(graph, n_iterations=LEIDEN_N_ITERATIONS, seed=LEIDEN_SEED)
_runtime_d = time.perf_counter() - _t0
print(f"leiden_community_detection: {clusters['community'].nunique()} communities, "
      f"total runtime {_runtime_d:.1f}s")


build_graph: 10888 nodes, 185021 edges, 0.0s


Loaded edgelist from cache: /Users/lukajin/Library/Caches/pwiden_engine/edgelist_482d5aecc1cdf02e8406c760986be0de474f3ee46c843542b908517aac403c9c_thresh_0.6000.bin


leiden_community_detection: 1504 communities, total runtime 2.0s


In [8]:
# clusters['node_id'] is a POSITIONAL index into `sequences` (i.e. into
# scoreable_voters, in order) -- map back to peptide_voters' original row index via
# scoreable_voters['index'] (preserved by reset_index() above), matching the same
# node_id convention generate_leiden_train_folds.ipynb relies on.
node_to_original_index = scoreable_voters["index"].to_numpy()
cluster_qmap = pd.Series(index=peptide_voters.index, dtype="float64")
cluster_qmap.iloc[node_to_original_index[clusters["node_id"].to_numpy()]] = clusters["community"].to_numpy()

peptide_voters["cluster_qmap"] = cluster_qmap

report_clustering_diagnostics(peptide_voters, "cluster_qmap", _runtime_d)


[cluster_qmap] runtime: 2.02s
[cluster_qmap] coverage: 10888/12371 peptides scored (88.0%)
[cluster_qmap] cluster count: 1504
[cluster_qmap] cluster size distribution: min=1, median=1, max=791, mean=7.2
[cluster_qmap] largest cluster is 7.3% of scored peptides


## Save the voter column (plus the coverage helper columns, useful for notebook 05's diagnostics)

In [9]:
save_peptide_voters(peptide_voters)


saved /Users/lukajin/PycharmProjects/soamp/data/clustering/peptide_voters.parquet (12371 rows, columns: ['peptide_id', 'sequence', 'smiles', 'has_noncanonical', 'bond_type', 'is_linear', 'cluster_fingerprint', 'cluster_descriptor', 'qmap_input_sequence', 'qmap_scoreable', 'cluster_qmap'])
